# Database - Creation
    - Author: Santiago Avella. Git hub: https://github.com/TiagoMimi

In [1]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
# *Configure pandas to display all columns in DataFrame outputs*
pd.set_option('display.max_columns', None)

## Load the data

In [2]:
# Current working directory
script_path = Path.cwd()

# Find the project folder
project_path = script_path

while project_path.name != "saber11_db":
    project_path = project_path.parent


In [3]:
# The folder containing the TXT files
data_path = project_path / "data" / "joined" 

# List of TXT files
file_list = [
    data_path / "teacher_schedule.csv",
    data_path / "teacher_rank.csv",
    data_path / "teacher_employment.csv",
    data_path / "teacher_education.csv",
    data_path / "teacher_clei.csv",
    data_path / "teacher_age.csv",
    data_path / "teacher_academic_assignment.csv",
    data_path / "student_info.csv",
    data_path / "socioeconomic_info.csv",
    data_path / "school_info.csv",
    data_path / "result_info.csv"
]


# Choose the DataFrame name for each file
df_names = [
"df_teacher_schedule",
"df_teacher_rank",
"df_teacher_employment",
"df_teacher_education",
"df_teacher_clei",
"df_teacher_age",
"df_teacher_academic_assignment",
"df_student_info",
"df_socioeconomic_info",
"df_school_info",
"df_result_info",
]

# Read each CSV separately
dataframes = {}

for name, file in zip(df_names, file_list):
    dataframes[name] = pd.read_csv(file, sep=";")

df_teacher_schedule = dataframes["df_teacher_schedule"]
df_teacher_rank = dataframes["df_teacher_rank"]
df_teacher_employment = dataframes["df_teacher_employment"]
df_teacher_education = dataframes["df_teacher_education"]
df_teacher_clei = dataframes["df_teacher_clei"]
df_teacher_age = dataframes["df_teacher_age"]
df_teacher_academic_assignment = dataframes["df_teacher_academic_assignment"]
df_student_info = dataframes["df_student_info"]
df_socioeconomic_info = dataframes["df_socioeconomic_info"]
df_school_info = dataframes["df_school_info"]
df_result_info = dataframes["df_result_info"]

## Clean primary and foreing key

### result saber11

#### School_info

In [4]:
columns = [
    "cole_cod_dane_establecimiento",
    "cole_cod_dane_sede",
    "cole_cod_depto_ubicacion",
    "cole_cod_mcpio_ubicacion",
    "cole_codigo_icfes"
]

for col in columns:
    df_school_info[col] = (
        df_school_info[col]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )


df_school_info = df_school_info.rename(columns={"cole_cod_dane_establecimiento":"school_institution_id",
                                                "cole_cod_dane_sede":"school_campus_id"})


#### Student_info

In [5]:
columns = [
    "cole_cod_dane_establecimiento",
    "cole_cod_dane_sede",
    "estu_cod_reside_depto",
    "estu_cod_reside_mcpio"]

for col in columns:
    df_student_info[col] = (
        df_student_info[col]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )


df_student_info = df_student_info.rename(columns={"cole_cod_dane_establecimiento":"school_institution_id",
                                                "cole_cod_dane_sede":"school_campus_id"})

#### socioeconomic_info

In [6]:
columns = [
    "cole_cod_dane_establecimiento",
    "cole_cod_dane_sede"]

for col in columns:
    df_socioeconomic_info[col] = (
        df_socioeconomic_info[col]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )


df_socioeconomic_info = df_socioeconomic_info.rename(columns={"cole_cod_dane_establecimiento":"school_institution_id",
                                                "cole_cod_dane_sede":"school_campus_id"})

#### Result_info

In [7]:
columns = [
    "cole_cod_dane_establecimiento",
    "cole_cod_dane_sede",    
    "estu_cod_depto_presentacion",
    "estu_cod_mcpio_presentacion"]

for col in columns:
    df_result_info[col] = (
        df_result_info[col]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )


df_result_info = df_result_info.rename(columns={"cole_cod_dane_establecimiento":"school_institution_id",
                                                "cole_cod_dane_sede":"school_campus_id"})

### teacher_info

In [16]:
teacher_dataframes = [
    df_teacher_schedule,
    df_teacher_rank,
    df_teacher_employment,
    df_teacher_education,
    df_teacher_clei,
    df_teacher_age,
    df_teacher_academic_assignment
]

for df in teacher_dataframes:

    if "SEDE_CODIGO" in df.columns:

        df.rename(
            columns={"SEDE_CODIGO": "school_campus_id"},
            inplace=True
        )

        df["school_campus_id"] = df["school_campus_id"].astype("string")

## Data Base

In [23]:
import sqlite3

In [24]:
database_path = Path("data/database/education.db")

database_path.parent.mkdir(
    parents=True,
    exist_ok=True
)
conn = sqlite3.connect(database_path)

In [25]:
conn.execute(
    "PRAGMA foreign_keys = ON;"
)

In [27]:
conn.executescript("""

-- SCHOOL

CREATE TABLE IF NOT EXISTS school_info (

    school_campus_id TEXT PRIMARY KEY

);


-- STUDENT

CREATE TABLE IF NOT EXISTS student_info (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS socioeconomic_info (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS result_info (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


-- TEACHER

CREATE TABLE IF NOT EXISTS teacher_schedule (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS teacher_rank (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS teacher_employment (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS teacher_education (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS teacher_clei (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS teacher_age (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);


CREATE TABLE IF NOT EXISTS teacher_academic_assignment (

    school_campus_id TEXT,

    FOREIGN KEY (school_campus_id)
        REFERENCES school_info(school_campus_id)

);

""")

conn.commit()